# 07. Prompt Building & Recipe Generation (RAG)

This notebook describes the **core RAG pipeline** used by CookMate:

1. We represent the user request with a `UserRequest` dataclass.
2. We retrieve similar recipes based on the pantry and preferences.
3. We build a **RAG prompt** combining:
   - the user request,
   - the retrieved recipes.
4. We call a **local LLaMA 3 model** via Ollama to generate a recipe.
5. We validate that the model output is **valid JSON** with a fixed schema.

The actual implementation lives in:

- `rag_pipeline/prompt_builder.py`
- `rag_pipeline/generator.py`

This notebook demonstrates how those pieces fit together.

In [10]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added: ", project_root)

Project root added:  /Users/biancaleoveanu/CookMate-Recipe-Generator


In [11]:
import json

from rag_pipeline.prompt_builder import UserRequest, build_rag_prompt
from rag_pipeline.search import search_recipes
from rag_pipeline.generator import generate_recipe

## 1. UserRequest and retrieval

We start by constructing a `UserRequest` and retrieving candidate recipes.

In [12]:
user = UserRequest(
    ingredients=["chicken", "rice", "onion"],
    diet="vegetarian",
    cuisine="Italian",
)

user

UserRequest(ingredients=['chicken', 'rice', 'onion'], diet='vegetarian', cuisine='Italian')

In [14]:
from rag_pipeline.query_builder import build_query

query = build_query(
    ingredients=user.ingredients,
    diet=user.diet,
    cuisine=user.cuisine,
)
retrieved = search_recipes(
    query,
    k=3,
)

len(retrieved)

3

In [15]:
retrieved[0]

{'recipe_id': 362239,
 'title': 'Italian Chicken and Rice',
 'ingredients_list': ['chicken',
  'onion',
  'chicken broth',
  'tomatoes',
  'French style green beans',
  'parmesan cheese'],
 'ingredients_structured': [{'ingredient': 'chicken', 'quantity': '12'},
  {'ingredient': 'onion', 'quantity': '1/2'},
  {'ingredient': 'chicken broth', 'quantity': '1'},
  {'ingredient': 'tomatoes', 'quantity': '1'},
  {'ingredient': 'French style green beans', 'quantity': '1 1/2'},
  {'ingredient': 'parmesan cheese', 'quantity': '1'}],
 'steps_list': ['Cut chicken into strips (I cut into more bite-size pieces).',
  'Coat large skillet with non-stick cooking spray and heat over medium-high.',
  'Add chicken and onion and cook approximately 3 minutes until chicken is no longer pink.',
  'Stir in chicken broth and undrained tomatoes.',
  'Bring to a boil.',
  'Stir in rice, beans and spices and reduce heat to simmer.',
  'Cover and cook for 5 minutes.',
  'Remove from heat and let stand 3 minutes, cov

## 2. Building the RAG prompt

We now build the final prompt that will be sent to the LLM.

The prompt includes:

- a concise description of the user request,
- a short summary of each retrieved recipe (title, ingredients, steps),
- explicit instructions and a **JSON schema** that the model must follow.

In [16]:
prompt = build_rag_prompt(user, retrieved)
print(prompt[:2000])  

You are CookMate, an AI cooking assistant that ALWAYS follows rules and ALWAYS outputs valid JSON.

###############
###  TASK   ###
###############

Your job is to generate a NEW recipe using:

- The user's pantry ingredients
- The user's diet (vegan, vegetarian, gluten-free, or none)
- The user's cuisine preference
- The retrieved recipe examples provided below

You MUST:
- Follow the JSON schema exactly (no missing fields, no extra fields).
- Respect all dietary restrictions.
- Use ONLY ingredients allowed by the user (and ingredients that fit the diet).
- Never hallucinate ingredients outside the allowed list.
- Use the retrieved recipes as inspiration for structure, timing, and flavor.

########################################
###  ADDITIONAL STRICT RULES (IMPORTANT)
########################################

- You MUST use ONLY the ingredients from the user's pantry list.
- If the pantry is small, create a simple recipe rather than inventing extra ingredients.
- You MAY reuse ingre

### JSON schema used in our prompt

The model must output a single JSON object with the following structure:

```json
{
  "title": "string",
  "ingredients": [
    {
      "item": "chicken breast",
      "quantity": 200,
      "unit": "g"
    }
  ],
  "steps": ["Step 1 ...", "Step 2 ..."],
  "time_minutes": 30,
  "servings": 2,
  "diet": "vegetarian",
  "cuisine": "Italian",
  "reason": "Short explanation of recipe design choices"
}

## 3. Running the full RAG pipeline

Next, we call `generate_recipe(...)` which:

1. Builds the query and retrieves recipes.
2. Estimates nutrition from retrieved recipes.
3. Builds the RAG prompt.
4. Calls the local LLaMA model via Ollama.
5. Validates the JSON and optionally retries if it's invalid.

In [17]:
result = generate_recipe(
    pantry=["chicken", "rice", "onion"],
    diet="vegetarian",
    cuisine="Italian",
    k=3,
    max_retries=1,
)

result["success"]

True

In [18]:
if result["success"]:
    recipe = result["recipe"]
    print(json.dumps(recipe, indent=2, ensure_ascii=False))
else:
    print("Generation failed.")
    print("Error:", result.get("error"))
    print("Validation message:", result.get("validation_message"))

{
  "title": "Vegetarian Italian Rice Bowl",
  "ingredients": [
    {
      "item": "rice",
      "quantity": "1 cup"
    },
    {
      "item": "onion",
      "quantity": "1 small"
    }
  ],
  "steps": [
    "Heat a large skillet over medium-high heat.",
    "Add the onion and cook until translucent, about 3-4 minutes.",
    "Add cooked rice to the skillet and stir-fry for about 2 minutes.",
    "Season with salt and pepper to taste."
  ],
  "time_minutes": 10,
  "servings": 1,
  "diet": "vegetarian",
  "cuisine": "Italian",
  "reason": "This recipe respects the user's vegetarian diet and Italian cuisine preference, using only ingredients from their pantry list.",
  "nutrition": {
    "calories": 568.1,
    "fat": 8.600000000000001,
    "carbs": 96.23333333333333,
    "protein": 23.03333333333333
  }
}


## 4. Inspecting ingredients with quantities

We verify that each ingredient has `item`, `quantity`, and `unit`.

In [19]:
if result["success"]:
    for ing in recipe.get("ingredients", []):
        print(ing)

{'item': 'rice', 'quantity': '1 cup'}
{'item': 'onion', 'quantity': '1 small'}


## 5. Nutrition estimate

If the RAG pipeline successfully estimated nutrition from retrieved recipes,
the generator attaches a `nutrition` field to the recipe:

```python
{
  "nutrition": {
      "calories": ...,
      "protein": ...,
      "carbs": ...,
      "fat": ...
  }
}

In [20]:
if result["success"]:
    nutrition = recipe.get("nutrition", {})
    nutrition